# 🇪🇺 Eurostat Health Data Access

This notebook demonstrates how to access European Union health statistics from **Eurostat**, the statistical office of the European Union.

## Overview

Eurostat provides comprehensive health data covering:
- **27 EU Member States** (post-Brexit)
- **EFTA countries** (Iceland, Norway, Switzerland, Liechtenstein)
- **Candidate countries** (Turkey, Serbia, Ukraine, etc.)

### Data Available
- Healthcare expenditure and financing
- Mortality and causes of death
- Health status indicators
- Health workforce (physicians, nurses)
- Hospital beds and facilities
- Health determinants (lifestyle, environment)

**Website:** https://ec.europa.eu/eurostat/web/health

**API Docs:** https://ec.europa.eu/eurostat/web/sdmx-web-services

## 📦 Setup

In [14]:
import matplotlib.pyplot as plt

from epidatasets.sources.eurostat import EurostatAccessor

# Initialize accessor
eurostat = EurostatAccessor()

print("✅ Eurostat Accessor initialized successfully!")

INFO:accessors.eurostat:Using eurostat Python library


✅ Eurostat Accessor initialized successfully!


## 🌍 Exploring Eurostat Coverage

Eurostat covers EU member states plus EFTA and candidate countries:

In [2]:
# List EU member states
eu_countries = eurostat.list_eu_countries()
print(f"🇪🇺 Total EU Member States: {len(eu_countries)}")
print("\nFirst 10 EU countries:")
eu_countries.head(10)

🇪🇺 Total EU Member States: 27

First 10 EU countries:


,country_code,country_name
0,AUT,Austria
1,BEL,Belgium
2,BGR,Bulgaria
3,HRV,Croatia
4,CYP,Cyprus
5,CZE,Czech Republic
6,DNK,Denmark
7,EST,Estonia
8,FIN,Finland
9,FRA,France


In [3]:
# List all countries (EU + EFTA + Candidates)
all_countries = eurostat.list_all_countries()

print("\n🌍 Countries by Type:")
for country_type in all_countries['type'].unique():
    subset = all_countries[all_countries['type'] == country_type]
    print(f"\n{country_type} ({len(subset)} countries):")
    print(f"  {', '.join(subset['country_code'].tolist())}")


🌍 Countries by Type:

EU Member (27 countries):
  AUT, BEL, BGR, HRV, CYP, CZE, DNK, EST, FIN, FRA, DEU, GRC, HUN, IRL, ITA, LVA, LTU, LUX, MLT, NLD, POL, PRT, ROU, SVK, SVN, ESP, SWE

EFTA (4 countries):
  ISL, LIE, NOR, CHE

Candidate (9 countries):
  ALB, MNE, MKD, SRB, TUR, UKR, MDA, BIH, GEO

Former EU (1 countries):
  GBR


## 📊 Available Health Indicators

In [4]:
# List common health indicators
indicators = eurostat.list_indicators()
print(f"📈 Total health indicators: {len(indicators)}")
print("\nAvailable indicators:")
indicators

📈 Total health indicators: 25

Available indicators:


,indicator_code,description
0,hlth_silc_01,"Self-perceived health by sex, age and income"
1,hlth_silc_11,Self-reported unmet needs for medical care
2,hlth_silc_12,Self-reported unmet needs for dental care
3,hlth_silc_20,"Body mass index by sex, age and income"
4,hlth_ehis_de2,"Healthy life years by sex, age and NUTS"
5,hlth_cd_aro,Causes of death - absolute numbers
6,hlth_cd_acdr2,Avoidable and treatable mortality by NUTS 2
7,demo_mlexpec,Life expectancy by age and sex
8,demo_pjanind__indic_de,Infant mortality rate
9,hlth_sha11_hf,Health care expenditure by financing scheme


In [5]:
# Search for specific indicators
print("🔍 Searching for 'mortality' indicators:")
mortality_indicators = eurostat.search_indicators('mortality')
mortality_indicators

🔍 Searching for 'mortality' indicators:


,indicator_code,description
6,hlth_cd_acdr2,Avoidable and treatable mortality by NUTS 2
8,demo_pjanind__indic_de,Infant mortality rate
23,demo_pjanind__covid,Excess mortality during COVID-19


In [6]:
# List causes of death
causes = eurostat.list_causes_of_death()
print("☠️  Causes of Death (ICD-10 based):")
causes

☠️  Causes of Death (ICD-10 based):


,cause_code,description
0,TOTAL,All causes of death
1,A_R,Infectious and parasitic diseases
2,COVID-19,COVID-19
3,B20_B24,HIV disease
4,C,Malignant neoplasms (cancer)
5,E,"Endocrine, nutritional and metabolic diseases"
6,E10_E14,Diabetes mellitus
7,F,Mental and behavioural disorders
8,G_H,Diseases of the nervous system and sense organs
9,I,Diseases of the circulatory system


## 💰 Healthcare Expenditure

Get healthcare spending data for EU countries.

In [7]:
# Get healthcare expenditure for selected countries
# Note: This requires internet connection and API access

try:
    expenditure = eurostat.get_healthcare_expenditure(
        countries=['DEU', 'FRA', 'ITA', 'ESP', 'NLD'],
        years=[2019, 2020, 2021, 2022]
    )

    if not expenditure.empty:
        print(f"Retrieved {len(expenditure)} records")
        print("\nSample data:")
        print(expenditure.head(10))
    else:
        print("No data retrieved. This may be due to API limitations.")

except Exception as e:
    print(f"Note: API access may be limited: {e}")
    print("The accessor is properly configured and will work when API is available.")

INFO:accessors.eurostat:Fetching indicator hlth_sha11_hf for 5 countries


Retrieved 7190 records

Sample data:
  freq     unit icha11_hf geo\TIME_PERIOD     1992     1993     1994     1995  \
0    A  EUR_HAB       HF1              AT      NaN      NaN      NaN  1580.22   
1    A  EUR_HAB       HF1              BA      NaN      NaN      NaN      NaN   
2    A  EUR_HAB       HF1              BE      NaN      NaN      NaN      NaN   
3    A  EUR_HAB       HF1              BG      NaN      NaN      NaN      NaN   
4    A  EUR_HAB       HF1              CH      NaN      NaN      NaN      NaN   
5    A  EUR_HAB       HF1              CY      NaN      NaN      NaN      NaN   
6    A  EUR_HAB       HF1              CZ      NaN      NaN      NaN      NaN   
7    A  EUR_HAB       HF1              DE  1463.01  1546.99  1672.72  1850.56   
8    A  EUR_HAB       HF1              DK      NaN      NaN      NaN      NaN   
9    A  EUR_HAB       HF1            EA19      NaN      NaN      NaN      NaN   

      1996     1997  ...     2015     2016     2017     2018     2019  

## 📉 Mortality Data

In [8]:
# Get mortality data by cause
try:
    mortality = eurostat.get_mortality_data(
        cause_code='COVID-19',
        countries=['DEU', 'FRA', 'ITA', 'ESP'],
        years=[2020, 2021, 2022]
    )

    if not mortality.empty:
        print(f"\nRetrieved {len(mortality)} mortality records")
        print("\nSample data:")
        print(mortality.head())
    else:
        print("No mortality data retrieved from API")

except Exception as e:
    print(f"Note: {e}")

INFO:accessors.eurostat:Fetching indicator hlth_cd_aro for 4 countries



Retrieved 700128 mortality records

Sample data:
  freq unit sex    age        icd10   resid geo\TIME_PERIOD  2011  2012  2013  \
0    A   NR   F  TOTAL  A15-A19_B90  TOT_IN              AT  20.0  12.0  21.0   
1    A   NR   F  TOTAL  A15-A19_B90  TOT_IN              BE  22.0  19.0  21.0   
2    A   NR   F  TOTAL  A15-A19_B90  TOT_IN              BG  32.0  33.0  32.0   
3    A   NR   F  TOTAL  A15-A19_B90  TOT_IN              CH  10.0  13.0  13.0   
4    A   NR   F  TOTAL  A15-A19_B90  TOT_IN              CY   1.0   2.0   1.0   

   ...  2015  2016  2017  2018  2019  2020  2021  2022  2023  2024  
0  ...  23.0  13.0  16.0  19.0  18.0  14.0  11.0  14.0  15.0  10.0  
1  ...  24.0  15.0   8.0  15.0  18.0  17.0   6.0   4.0   9.0   NaN  
2  ...  30.0  11.0  21.0  26.0  21.0  14.0  16.0  10.0  14.0   NaN  
3  ...   7.0  11.0   8.0  11.0   4.0   7.0   6.0   4.0   6.0   NaN  
4  ...   1.0   0.0   1.0   1.0   0.0   0.0   2.0   3.0   2.0   NaN  

[5 rows x 21 columns]


## 👨‍⚕️ Health Workforce

In [9]:
# Get physician data
try:
    physicians = eurostat.get_physicians(
        countries=['DEU', 'FRA', 'ITA', 'ESP', 'POL'],
        years=[2020, 2021, 2022]
    )

    if not physicians.empty:
        print("Physicians Data:")
        print(physicians.head(10))

except Exception as e:
    print(f"Note: {e}")

INFO:accessors.eurostat:Fetching indicator hlth_rs_prsrg for 5 countries


Physicians Data:
  freq   unit isco08 geo\TIME_PERIOD    1993    1994    1995    1996    1997  \
0    A  HAB_P  OC221              AL     NaN     NaN     NaN     NaN     NaN   
1    A  HAB_P  OC221             AL0     NaN     NaN     NaN     NaN     NaN   
2    A  HAB_P  OC221              AT  303.47  293.51  284.65  279.37  273.54   
3    A  HAB_P  OC221            AT11  449.46  429.32  414.46  415.10  404.36   
4    A  HAB_P  OC221            AT12  407.41  390.38  380.26  367.03  353.38   
5    A  HAB_P  OC221            AT13  175.46  169.10  164.15  161.88  160.61   
6    A  HAB_P  OC221            AT21  346.08  342.85  336.90  329.83  325.38   
7    A  HAB_P  OC221            AT22  323.28  313.62  305.63  300.09  293.73   
8    A  HAB_P  OC221            AT31  419.50  410.58  394.60  387.44  373.45   
9    A  HAB_P  OC221            AT32  324.42  316.56  304.77  299.67  290.88   

     1998  ...    2012    2013    2014    2015    2016    2017    2018  \
0     NaN  ...  784.74  780.

In [10]:
# Get hospital beds data
try:
    beds = eurostat.get_hospital_beds(
        countries=['DEU', 'FRA', 'ITA'],
        years=[2020, 2021]
    )

    if not beds.empty:
        print("Hospital Beds Data:")
        print(beds.head(10))

except Exception as e:
    print(f"Note: {e}")

INFO:accessors.eurostat:Fetching indicator hlth_rs_bdsrg for 3 countries


Hospital Beds Data:
  freq   unit   facility geo\TIME_PERIOD     1993     1994     1995     1996  \
0    A  HAB_P  HBEDI_PSY              AL      NaN      NaN      NaN      NaN   
1    A  HAB_P  HBEDI_PSY             AL0      NaN      NaN      NaN      NaN   
2    A  HAB_P  HBEDI_PSY              AT  1294.31  1356.60  1452.01  1589.26   
3    A  HAB_P  HBEDI_PSY            AT11      NaN      NaN      NaN      NaN   
4    A  HAB_P  HBEDI_PSY            AT12  1296.84  1338.73  1361.37  1417.23   
5    A  HAB_P  HBEDI_PSY            AT13  1035.20  1040.41  1042.51  1142.03   
6    A  HAB_P  HBEDI_PSY            AT21  1256.01  1510.03  1603.65  1755.30   
7    A  HAB_P  HBEDI_PSY            AT22  5513.78  5594.92  5620.08  5809.13   
8    A  HAB_P  HBEDI_PSY            AT31  1239.27  1274.93  1659.72  1767.35   
9    A  HAB_P  HBEDI_PSY            AT32   924.78   967.89   977.73  1076.40   

      1997     1998  ...      2012     2013     2014     2015  2016  2017  \
0      NaN      NaN  .

## 📊 Life Expectancy

In [11]:
# Get life expectancy data
try:
    life_exp = eurostat.get_life_expectancy(
        countries=['DEU', 'FRA', 'ITA', 'ESP', 'POL', 'SWE'],
        years=[2019, 2020, 2021, 2022]
    )

    if not life_exp.empty:
        print("Life Expectancy Data:")
        print(life_exp.head(10))

        # Compare across countries
        comparison = eurostat.compare_countries(
            indicator_code='demo_mlexpec',
            countries=['DEU', 'FRA', 'ITA', 'ESP', 'POL', 'SWE'],
            years=[2019, 2020, 2021, 2022]
        )
        print("\nComparison Table:")
        print(comparison)

except Exception as e:
    print(f"Note: {e}")

INFO:accessors.eurostat:Fetching indicator demo_mlexpec for 6 countries
INFO:accessors.eurostat:Fetching indicator demo_mlexpec for 6 countries


Life Expectancy Data:
  freq unit sex age geo\TIME_PERIOD  1960  1961  1962  1963  1964  ...  2015  \
0    A   YR   F  Y1              AL   NaN   NaN   NaN   NaN   NaN  ...  79.2   
1    A   YR   F  Y1              AM   NaN   NaN   NaN   NaN   NaN  ...  77.9   
2    A   YR   F  Y1              AT   NaN   NaN   NaN   NaN   NaN  ...  83.0   
3    A   YR   F  Y1              AZ   NaN   NaN   NaN   NaN   NaN  ...  77.5   
4    A   YR   F  Y1              BE  73.7  74.5  74.0  73.9  74.6  ...  82.6   
5    A   YR   F  Y1              BG  73.2  73.8  73.0  73.6  74.5  ...  77.6   
6    A   YR   F  Y1              BY   NaN   NaN   NaN   NaN   NaN  ...  78.2   
7    A   YR   F  Y1              CH  74.5  75.0  74.6  74.6  75.4  ...  84.4   
8    A   YR   F  Y1              CY   NaN   NaN   NaN   NaN   NaN  ...  82.9   
9    A   YR   F  Y1              CZ  73.7  73.8  73.3  73.9  74.0  ...  80.8   

   2016  2017  2018  2019  2020  2021  2022  2023  2024  
0  79.8  79.7  80.2  80.4  79.4  77.2  

## 📈 Visualization Example

In [15]:
# Visualize life expectancy comparison (if data available)
if not life_exp.empty and 'value' in life_exp.columns:
    fig, ax = plt.subplots(figsize=(12, 6))

    for country in life_exp['geo'].unique():
        country_data = life_exp[life_exp['geo'] == country]
        ax.plot(country_data['time'], country_data['value'],
                marker='o', label=country, linewidth=2)

    ax.set_xlabel('Year', fontsize=12)
    ax.set_ylabel('Life Expectancy (years)', fontsize=12)
    ax.set_title('Life Expectancy at Birth - Selected EU Countries', fontsize=14)
    ax.legend(title='Country', bbox_to_anchor=(1.05, 1), loc='upper left')
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

## 🏥 Self-Perceived Health

In [16]:
# Get self-perceived health status
try:
    self_health = eurostat.get_self_perceived_health(
        countries=['DEU', 'FRA', 'ITA', 'ESP'],
        years=[2021, 2022]
    )

    if not self_health.empty:
        print("Self-Perceived Health Data:")
        print(self_health.head(10))

except Exception as e:
    print(f"Note: {e}")

INFO:accessors.eurostat:Fetching indicator hlth_silc_01 for 4 countries


Self-Perceived Health Data:
  freq unit wstatus     age sex levels geo\TIME_PERIOD  2008  2009  2010  ...  \
0    A   PC     EMP  Y16-24   F    BAD              AL   NaN   NaN   NaN  ...   
1    A   PC     EMP  Y16-24   F    BAD              AT   0.3   0.5   0.9  ...   
2    A   PC     EMP  Y16-24   F    BAD              BE   2.0   0.7   2.7  ...   
3    A   PC     EMP  Y16-24   F    BAD              BG   0.7   0.0   0.0  ...   
4    A   PC     EMP  Y16-24   F    BAD              CH   1.0   0.0   0.2  ...   
5    A   PC     EMP  Y16-24   F    BAD              CY   0.0   0.0   0.0  ...   
6    A   PC     EMP  Y16-24   F    BAD              CZ   0.1   0.0   0.0  ...   
7    A   PC     EMP  Y16-24   F    BAD              DE   0.4   0.3   1.0  ...   
8    A   PC     EMP  Y16-24   F    BAD              DK   2.7   0.6   4.0  ...   
9    A   PC     EMP  Y16-24   F    BAD              EA   0.7   0.9   1.1  ...   

   2016  2017  2018  2019  2020  2021  2022  2023  2024  2025  
0   NaN   0.0   

## 🔍 Advanced Usage: Generic Indicator Access

Access any Eurostat indicator by its code:

In [17]:
# Get data dictionary for an indicator
try:
    metadata = eurostat.get_data_dictionary('demo_mlexpec')
    print("Indicator Metadata:")
    for key, value in metadata.items():
        print(f"  {key}: {value}")

except Exception as e:
    print(f"Note: {e}")

ERROR:accessors.eurostat:Error fetching data dictionary: 404 Client Error:  for url: https://ec.europa.eu/eurostat/wdds/rest/data/v2.1/json/en/demo_mlexpec


Indicator Metadata:


In [18]:
# Get any indicator by code
try:
    # Example: Infant mortality rate
    infant_mort = eurostat.get_infant_mortality(
        countries=['DEU', 'FRA', 'ITA'],
        years=[2019, 2020, 2021]
    )

    if not infant_mort.empty:
        print("Infant Mortality Rate:")
        print(infant_mort.head())

except Exception as e:
    print(f"Note: {e}")

INFO:accessors.eurostat:Fetching indicator demo_pjanind__indic_de for 3 countries
ERROR:accessors.eurostat:Error using eurostat library: 'NoneType' object has no attribute 'get'
INFO:accessors.eurostat:Falling back to REST API
INFO:accessors.eurostat:Fetching from API: https://ec.europa.eu/eurostat/wdds/rest/data/v2.1/json/en/demo_pjanind__indic_de
ERROR:accessors.eurostat:Error fetching from API: 404 Client Error:  for url: https://ec.europa.eu/eurostat/wdds/rest/data/v2.1/json/en/demo_pjanind__indic_de?geo=DE%2CFR%2CIT&time=2019%2C2020%2C2021


## 🔧 Using the eurostat Library (Optional)

For better performance, install the `eurostat` Python library:

```bash
pip install eurostat
```

The accessor will automatically use the library if available.

In [ ]:
# Check if eurostat library is available
from epidatasets.sources.eurostat import EUROSTAT_LIB_AVAILABLE

if EUROSTAT_LIB_AVAILABLE:
    print("✅ eurostat library is installed and will be used")
else:
    print("⚠️  eurostat library not installed")
    print("   Install with: pip install eurostat")
    print("   Currently using REST API fallback")

## 📚 References

- **Eurostat Health:** https://ec.europa.eu/eurostat/web/health
- **Data Browser:** https://ec.europa.eu/eurostat/databrowser
- **API Documentation:** https://ec.europa.eu/eurostat/web/sdmx-web-services
- **Python Library:** https://pypi.org/project/eurostat/

## 📝 Notes

1. **Data Availability:** Some indicators may have missing data for certain countries/years.
2. **API Limits:** Eurostat API has rate limits; consider caching results for repeated use.
3. **Country Codes:** Use ISO3 codes (e.g., 'DEU' for Germany) or ISO2 (e.g., 'DE').
4. **Time Coverage:** Most indicators have annual data from 2000 onwards.